# 01 · Define a study

```
      ((♥))          ((♥))
       /|\   ~ ♥ ~    /|\        a study is: a place, a projection,
      /_|_\  ~ ♥ ~   /_|_\       a band plan, and the layers you have
```

Everything in this repo starts from a **study**: *where* you are twinning, *in which
projection*, *which signals* you care about, and *which geospatial layers* you have (or
need to go get). The pipeline reads that from a versioned `study.toml`, which
`ulap-scope init` writes interactively.

This notebook uses the same code the wizard does — `ulap_scope.study` — but from Python,
so you can script a study, generate a batch of them, or check one into review.

**Runs anywhere.** Pure standard library: no GIS stack, no Blender, no Sionna.

In [ ]:
# Make `ulap_demo` (examples/) and `ulap_scope` (the pipeline package) importable
# straight from a fresh checkout -- no install required.
import sys, pathlib
here = pathlib.Path.cwd()
EXAMPLES = next(p for p in [here, *here.parents] if (p / "ulap_demo").is_dir())
sys.path[:0] = [str(EXAMPLES), str(EXAMPLES.parent / "ulap-scope")]
print("examples:", EXAMPLES)

In [ ]:
from ulap_scope.study import (StudySpec, BAND_CATALOG, LAYER_FALLBACKS,
                               NON_METRIC_EPSG, bbox_deg, utm_epsg, utm_zone,
                               to_toml, load_study, save_study)

pilot = StudySpec()          # defaults reproduce the Barbados pilot
print(pilot.name, "@", pilot.lon, pilot.lat)
print("bbox (min_lon, min_lat, max_lon, max_lat):")
print("  ", [round(v, 6) for v in pilot.bbox])
print("study box:", 2 * pilot.half_width_m, "m across")

## The projection trap

The single most expensive mistake in an RF digital twin is **building it in a projection
that isn't metres**. Ray tracers are happy to trace a scene in Web Mercator; the geometry
is simply wrong, and every link budget quietly inherits the error.

`utm_epsg()` computes the right metric CRS for a longitude/latitude, and the spec knows
which codes to warn about:

In [ ]:
for name, lon, lat in [("Newton, Barbados", -59.533908, 13.088121),
                       ("Nairobi, Kenya",     36.8219,   -1.2921),
                       ("Manila, Philippines", 120.9842,  14.5995),
                       ("Reykjavik, Iceland",  -21.9426,  64.1466)]:
    zone, hemi = utm_zone(lon, lat)
    print(f"{name:<22} UTM {zone}{hemi}  ->  EPSG:{utm_epsg(lon, lat)}")

print()
for epsg, why in NON_METRIC_EPSG.items():
    print(f"EPSG:{epsg} is a trap — {why}")

How bad is "2.6 % at 13° N", concretely? Web Mercator stretches distance by
`1 / cos(latitude)`. On a 1 km link that is metres of error — small. On a **path-loss
exponent fitted over a 2 km transect**, it is a systematic bias in the one number you
were trying to measure.

In [ ]:
import math

for lat in (0, 13.09, 35, 55):
    stretch = 1 / math.cos(math.radians(lat))
    err_m = (stretch - 1) * 1000
    # 20*log10 of the distance error is the path-loss error it induces
    err_db = 20 * math.log10(stretch)
    print(f"lat {lat:>5.1f}°  scale {stretch:.4f}  "
          f"({err_m:6.1f} m per km, {err_db:.2f} dB of phantom path loss)")

## Defining a study somewhere new

Say you want to twin a 3 km box around **Nairobi**. Give it a centre and a half-width;
the projection, bounding box and defaults follow.

In [ ]:
nairobi = StudySpec(
    name="nairobi-cbd",
    lon=36.8219, lat=-1.2921,
    half_width_m=1500.0,                     # 3 km box
    epsg=utm_epsg(36.8219, -1.2921),         # let the geodesy pick the CRS
    bands_ghz=(1.8, 3.5, 26.0),
    primary_band_ghz=3.5,
    mmwave_ghz=(26.0,),
    tx_power_dbm=36.0,
    rx_height_m=1.5,
).validate()

print("EPSG:", nairobi.epsg, "(recommended:", nairobi.recommended_epsg, ")")
print("bbox:", [round(v, 5) for v in nairobi.bbox])
print("CRS warning:", nairobi.crs_warning() or "none — metric CRS ✓")

### `validate()` fails loudly, on purpose

A study spec that is quietly wrong costs you a full pipeline run. Every constraint is
checked up front with a message that names the field and the range:

In [ ]:
bad_specs = [
    ("primary band not in the band list", dict(bands_ghz=(1.8, 3.5), primary_band_ghz=28.0)),
    ("receiver underground",               dict(rx_height_m=-1.0)),
    ("name with a space",                  dict(name="my study")),
    ("unknown output product",             dict(outputs=("coverage", "teleport"))),
]

for label, kwargs in bad_specs:
    try:
        StudySpec(**kwargs).validate()
    except ValueError as e:
        print(f"{label:<36} -> {e}")

## The band plan, and why 3.5 GHz keeps winning

`BAND_CATALOG` encodes what the pilot study measured, so the recommendation travels with
the code rather than living in a slide deck:

In [ ]:
for ghz, (label, note) in BAND_CATALOG.items():
    star = " ♥" if ghz == 3.5 else "  "
    print(f"{star} {ghz:>5} GHz  {label:<11} {note}")

## Layers: what you have, and what we recommend fetching

Most people do not have national LiDAR. The spec's job is to make the *fallback explicit*
and recorded, so a result is always traceable to the data that produced it.

In [ ]:
from ulap_scope.study import Layer

for layer, (source, description) in LAYER_FALLBACKS.items():
    print(f"{layer:<10} ♥ {source:<20} {description}")

# Mix local data with open fallbacks:
nairobi.layers["buildings"] = Layer(source="local", path="data/nairobi_footprints.gpkg")
nairobi.layers["terrain"] = Layer(source="copernicus-glo30")
nairobi.validate()
print("\nbuildings layer ->", nairobi.layers["buildings"])

## Write it, read it back

`study.toml` is the artefact you commit next to your results. Round-tripping it is how
you know a study is reproducible.

In [ ]:
from pathlib import Path

out = Path("nairobi-cbd.study.toml")
save_study(nairobi, out)
print(out.read_text())

In [ ]:
reloaded = load_study(out)
assert reloaded.name == nairobi.name
assert reloaded.epsg == nairobi.epsg
assert reloaded.bands_ghz == nairobi.bands_ghz
print("round-trip ok:", reloaded.name, reloaded.bands_ghz, "EPSG:", reloaded.epsg)

out.unlink()   # tidy up

## Compare with the shipped pilot study

`examples/data/newton-bbd.study.toml` is the Barbados pilot, written by
`ulap-scope init --defaults`:

In [ ]:
pilot_file = EXAMPLES / "data" / "newton-bbd.study.toml"
pilot = load_study(pilot_file)

print(f"{'field':<18} {'pilot (Newton)':<22} {'yours (Nairobi)'}")
print("-" * 62)
for field in ["name", "epsg", "half_width_m", "primary_band_ghz", "tx_power_dbm", "rx_height_m"]:
    print(f"{field:<18} {str(getattr(pilot, field)):<22} {getattr(nairobi, field)}")

## Next

- **Your turn:** change the coordinates above to somewhere you know, and keep the
  `study.toml`. That file is the input to `ulap-scope clip / preprocess / build`.
- Prefer a guided walk-through? Run the real wizard: `ulap-scope init`.
- Next notebook: **[02 · Link budget](02_link_budget.ipynb)** — real Sionna RT output
  versus a 30-line analytical model.